In [1]:
# Cell 1: Install all required packages
!pip install -q google-generativeai faiss-cpu rank-bm25 sentence-transformers \
    PyMuPDF tqdm pandas numpy scikit-learn langchain langchain-community \
    langchain-google-genai diskcache rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Cell 2: Mount Google Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, time, hashlib, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

PDF_FOLDER = "/content/drive/MyDrive/WDR_Reports"
CACHE_DIR  = "/content/wdr_cache"
OUTPUT_DIR = "/content/wdr_output"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify PDFs exist
pdf_files = sorted(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s):")
for p in pdf_files:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

Mounted at /content/drive
Found 10 PDF(s):
  World Development Report 2016.pdf  (10.2 MB)
  World Development Report 2017.pdf  (13.8 MB)
  World Development Report 2018.pdf  (9.7 MB)
  World Development Report 2019.pdf  (11.1 MB)
  World Development Report 2020.pdf  (20.2 MB)
  World Development Report 2021.pdf  (50.8 MB)
  World Development Report 2022.pdf  (6.2 MB)
  World Development Report 2023.pdf  (14.4 MB)
  World Development Report 2024.pdf  (9.2 MB)
  World Development Report 2025.pdf  (11.2 MB)


In [3]:
# Cell 3: Configure Gemini API with rate-limit-aware retry
import google.generativeai as genai
from google.colab import userdata
import time, re

GEMINI_API_KEY = userdata.get("Gemini_Key")
genai.configure(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-3.1-flash-lite"
generation_config = genai.GenerationConfig(
    temperature=0.2,
    max_output_tokens=1024,
)

FREE_TIER_DELAY = 4.5   # seconds between every API call
MAX_RETRIES     = 6

def call_gemini(prompt: str, system: str = ""):
    model = genai.GenerativeModel(
        model_name=MODEL_ID,
        generation_config=generation_config,
        system_instruction=system if system else None,
    )
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(FREE_TIER_DELAY)
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            err_str = str(e)
            match = re.search(r'retry in (\d+(?:\.\d+)?)s', err_str, re.IGNORECASE)
            wait  = float(match.group(1)) + 2 if match else (15 * (attempt + 1))
            if attempt < MAX_RETRIES - 1:
                tqdm.write(f"    [Rate limit] Waiting {wait:.0f}s before retry {attempt+2}/{MAX_RETRIES}...")
                time.sleep(wait)
            else:
                raise e

_ = call_gemini("Say OK.")
print(f"Gemini connected: {MODEL_ID} | Delay per call: {FREE_TIER_DELAY}s")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini connected: gemini-3.1-flash-lite | Delay per call: 4.5s


In [4]:
# Cell 4: Parse PDFs with page-level caching
import fitz  # PyMuPDF
import diskcache as dc

doc_cache = dc.Cache(os.path.join(CACHE_DIR, "parsed_docs"))

def parse_pdf_cached(pdf_path: Path):
    file_hash = hashlib.md5(pdf_path.read_bytes()).hexdigest()
    cache_key = f"parse_{file_hash}"

    if cache_key in doc_cache:
        return doc_cache[cache_key]

    pages = []
    doc = fitz.open(str(pdf_path))
    year_match = re.search(r"20[12]\d", pdf_path.stem)
    year = int(year_match.group()) if year_match else 0

    for page_num in tqdm(range(len(doc)), desc=f"  Parsing {pdf_path.name}", leave=False):
        page = doc[page_num]
        text = page.get_text("text").strip()
        if len(text) > 100:          # skip near-blank pages
            pages.append({
                "year":     year,
                "filename": pdf_path.name,
                "page":     page_num + 1,
                "text":     text,
            })
    doc.close()

    doc_cache[cache_key] = pages
    return pages

# Parse all PDFs
all_pages = []
for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
    pages = parse_pdf_cached(pdf)
    all_pages.extend(pages)

print(f"\nTotal pages extracted: {len(all_pages)}")

Parsing PDFs:   0%|          | 0/10 [00:00<?, ?it/s]

  Parsing World Development Report 2016.pdf:   0%|          | 0/359 [00:00<?, ?it/s]

  Parsing World Development Report 2017.pdf:   0%|          | 0/307 [00:00<?, ?it/s]

  Parsing World Development Report 2018.pdf:   0%|          | 0/239 [00:00<?, ?it/s]

  Parsing World Development Report 2019.pdf:   0%|          | 0/151 [00:00<?, ?it/s]

...

  Parsing World Development Report 2022.pdf:   0%|          | 0/281 [00:00<?, ?it/s]

  Parsing World Development Report 2023.pdf:   0%|          | 0/348 [00:00<?, ?it/s]

  Parsing World Development Report 2024.pdf:   0%|          | 0/276 [00:00<?, ?it/s]

  Parsing World Development Report 2025.pdf:   0%|          | 0/408 [00:00<?, ?it/s]


Total pages extracted: 2854


In [5]:
# Cell 5: Semantic / sliding-window chunking with overlap
CHUNK_SIZE    = 600    # tokens approx (chars / 4)
CHUNK_OVERLAP = 100

def chunk_page_text(page: dict, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    words = page["text"].split()
    chunks, start = [], 0
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])
        if len(chunk_text.strip()) > 80:
            chunks.append({
                **{k: page[k] for k in ("year", "filename", "page")},
                "chunk_id": f"{page['filename']}_p{page['page']}_c{len(chunks)}",
                "text":     chunk_text,
            })
        start += chunk_size - overlap
    return chunks

chunk_cache_path = os.path.join(CACHE_DIR, "chunks.pkl")

if os.path.exists(chunk_cache_path):
    with open(chunk_cache_path, "rb") as f:
        all_chunks = pickle.load(f)
    print(f"Loaded {len(all_chunks)} chunks from cache.")
else:
    all_chunks = []
    for page in tqdm(all_pages, desc="Chunking pages"):
        all_chunks.extend(chunk_page_text(page))
    with open(chunk_cache_path, "wb") as f:
        pickle.dump(all_chunks, f)
    print(f"Created and cached {len(all_chunks)} chunks.")

chunk_lengths = [len(c["text"].split()) for c in all_chunks]

print(f"\n{'='*45}")
print(f"  Total chunks         : {len(all_chunks)}")
print(f"  Avg chunk length     : {np.mean(chunk_lengths):.0f} words")

Chunking pages:   0%|          | 0/2854 [00:00<?, ?it/s]

Created and cached 4589 chunks.

  Total chunks         : 4589
  Avg chunk length     : 374 words


In [6]:
# Cell 6: Compute embeddings and build FAISS index (cached)
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL  = "all-MiniLM-L6-v2"
embed_cache  = os.path.join(CACHE_DIR, "embeddings.npy")
index_cache  = os.path.join(CACHE_DIR, "faiss.index")

embedder = SentenceTransformer(EMBED_MODEL)

if os.path.exists(embed_cache) and os.path.exists(index_cache):
    embeddings = np.load(embed_cache)
    faiss_index = faiss.read_index(index_cache)
    print(f"Loaded embeddings {embeddings.shape} and FAISS index from cache.")
else:
    texts = [c["text"] for c in all_chunks]
    batch_size = 256

    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding chunks"):
        batch = texts[i : i + batch_size]
        embeddings.append(embedder.encode(batch, show_progress_bar=False))
    embeddings = np.vstack(embeddings).astype("float32")

    faiss.normalize_L2(embeddings)
    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)

    np.save(embed_cache, embeddings)
    faiss.write_index(faiss_index, index_cache)
    print(f"Built FAISS index: {faiss_index.ntotal} vectors, dim={embeddings.shape[1]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks:   0%|          | 0/18 [00:00<?, ?it/s]

Built FAISS index: 4589 vectors, dim=384


In [7]:
# Cell 7: Build BM25 keyword index
from rank_bm25 import BM25Okapi

bm25_cache = os.path.join(CACHE_DIR, "bm25.pkl")

if os.path.exists(bm25_cache):
    with open(bm25_cache, "rb") as f:
        bm25 = pickle.load(f)
    print("Loaded BM25 index from cache.")
else:
    tokenized = [c["text"].lower().split() for c in tqdm(all_chunks, desc="Tokenizing for BM25")]
    bm25 = BM25Okapi(tokenized)
    with open(bm25_cache, "wb") as f:
        pickle.dump(bm25, f)
    print(f"Built BM25 index over {len(tokenized)} chunks.")

Tokenizing for BM25:   0%|          | 0/4589 [00:00<?, ?it/s]

Built BM25 index over 4589 chunks.


In [8]:
# Cell 8: Hybrid retrieval with cross-encoder reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_retrieve(query: str, top_k: int = 20, final_k: int = 5):
    # Dense
    q_emb = embedder.encode([query], show_progress_bar=False).astype("float32")
    faiss.normalize_L2(q_emb)
    _, dense_ids = faiss_index.search(q_emb, top_k)
    dense_ids = dense_ids[0].tolist()

    # Sparse
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    sparse_ids  = np.argsort(bm25_scores)[::-1][:top_k].tolist()

    # RRF fusion
    k_rrf = 60
    rrf_scores = {}
    for rank, idx in enumerate(dense_ids):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k_rrf + rank + 1)
    for rank, idx in enumerate(sparse_ids):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k_rrf + rank + 1)

    candidate_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

    # Cross-encoder rerank
    pairs = [(query, all_chunks[i]["text"]) for i in candidate_ids]
    ce_scores = reranker.predict(pairs, show_progress_bar=False)
    ranked    = sorted(zip(candidate_ids, ce_scores), key=lambda x: x[1], reverse=True)

    return [
        {**all_chunks[idx], "rerank_score": float(score)}
        for idx, score in ranked[:final_k]
    ]


def simple_retrieve(query: str, top_k: int = 5):
    q_emb = embedder.encode([query], show_progress_bar=False).astype("float32")
    faiss.normalize_L2(q_emb)
    _, ids = faiss_index.search(q_emb, top_k)
    return [all_chunks[i] for i in ids[0]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
# Cell 9: HyDE — Hypothetical Document Embedding for query expansion
def hyde_rewrite(query: str):
    prompt = (
        f"Write a concise, factual paragraph (≤120 words) that would be found in a "
        f"World Bank World Development Report and directly answers:\n\n'{query}'\n\n"
        f"Only write the passage, no preamble."
    )
    return call_gemini(prompt)

def enhanced_retrieve(query: str, use_hyde: bool = True, final_k: int = 5):
    retrieval_query = hyde_rewrite(query) if use_hyde else query
    return hybrid_retrieve(retrieval_query, final_k=final_k)

In [10]:
# Cell 10: Generation pipelines

SYSTEM_PROMPT = (
    "You are an expert analyst of World Bank World Development Reports (2016–2025). "
    "Answer ONLY using the provided context. Cite the report year and page number. "
    "If the context is insufficient, say so explicitly."
)

def format_context(chunks: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[WDR {c['year']} | Page {c['page']}]\n{c['text']}"
        for c in chunks
    )

def baseline_rag(query: str) -> dict:
    chunks  = simple_retrieve(query, top_k=5)
    context = format_context(chunks)
    prompt  = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    answer  = call_gemini(prompt, system=SYSTEM_PROMPT)
    return {"query": query, "answer": answer, "chunks": chunks, "pipeline": "baseline"}

def enhanced_rag(query: str) -> dict:
    chunks  = enhanced_retrieve(query, use_hyde=True, final_k=5)
    context = format_context(chunks)
    prompt  = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Provide a structured answer with: (1) direct answer, (2) supporting evidence "
        f"citing [WDR year | page], (3) any caveats.\n\nAnswer:"
    )
    answer  = call_gemini(prompt, system=SYSTEM_PROMPT)
    return {"query": query, "answer": answer, "chunks": chunks, "pipeline": "enhanced"}

In [11]:
# Cell 11: Diverse test query set (simple facts, deep context, ambiguous, edge cases)
TEST_QUERIES = [
    # Simple factual
    {"id": "Q1",  "type": "simple_fact",    "query": "What was the theme of the World Development Report 2016?"},
    {"id": "Q2",  "type": "simple_fact",    "query": "What year did WDR focus on governance and the law?"},
    {"id": "Q3",  "type": "simple_fact",    "query": "Which WDR report focused on the changing nature of work?"},
    # Deep context
    {"id": "Q4",  "type": "deep_context",   "query": "How has the World Bank's approach to digital dividends evolved between 2016 and 2021?"},
    {"id": "Q5",  "type": "deep_context",   "query": "What specific policy recommendations does the WDR make for human capital investment in developing countries?"},
    {"id": "Q6",  "type": "deep_context",   "query": "How do WDR reports address the role of institutions in economic development?"},
    # Ambiguous
    {"id": "Q7",  "type": "ambiguous",      "query": "What does the WDR say about poverty?"},
    {"id": "Q8",  "type": "ambiguous",      "query": "How does climate change affect development according to the World Bank?"},
    # Edge cases
    {"id": "Q9",  "type": "edge_case",      "query": "What is the GDP of Nigeria mentioned in any WDR?"},
    {"id": "Q10", "type": "edge_case",      "query": "Does the WDR 2025 mention quantum computing applications?"},
]

print(f"Test set: {len(TEST_QUERIES)} queries across {pd.Series([q['type'] for q in TEST_QUERIES]).value_counts().to_dict()}")

Test set: 10 queries across {'simple_fact': 3, 'deep_context': 3, 'ambiguous': 2, 'edge_case': 2}


In [12]:
# Cell 12: Run both pipelines with partial save on every query
import os, pickle, time
from tqdm.auto import tqdm

results_cache   = os.path.join(CACHE_DIR, "eval_results.pkl")
partial_cache   = os.path.join(CACHE_DIR, "eval_partial.pkl")

# Load any previously completed results (survive crashes/rate limits)
if os.path.exists(results_cache):
    with open(results_cache, "rb") as f:
        eval_results = pickle.load(f)
    print(f"Loaded {len(eval_results)} fully completed results from cache.")
else:
    if os.path.exists(partial_cache):
        with open(partial_cache, "rb") as f:
            eval_results = pickle.load(f)
        tqdm.write(f"Resuming from partial cache: {len(eval_results)} done.")
    else:
        eval_results = []

completed_ids = {r["id"] for r in eval_results}
remaining     = [q for q in TEST_QUERIES if q["id"] not in completed_ids]
print(f"Completed: {len(completed_ids)}/10  |  Remaining: {len(remaining)}")

for q in tqdm(remaining, desc="Evaluating queries"):
    tqdm.write(f"  [{q['id']}] {q['type']}: {q['query'][:60]}...")

    base = baseline_rag(q["query"])
    tqdm.write(f"    baseline done, sleeping...")
    time.sleep(FREE_TIER_DELAY)

    enha = enhanced_rag(q["query"])

    eval_results.append({
        "id":       q["id"],
        "type":     q["type"],
        "query":    q["query"],
        "baseline": base,
        "enhanced": enha,
    })

    # Save after every query — never lose progress
    with open(partial_cache, "wb") as f:
        pickle.dump(eval_results, f)
    tqdm.write(f"    [{q['id']}] saved. ({len(eval_results)}/10 done)")

# Promote partial → final cache when complete
if len(eval_results) == len(TEST_QUERIES):
    with open(results_cache, "wb") as f:
        pickle.dump(eval_results, f)
    tqdm.write("All queries complete. Final cache saved.")

Completed: 0/10  |  Remaining: 10


Evaluating queries:   0%|          | 0/10 [00:00<?, ?it/s]

  [Q1] simple_fact: What was the theme of the World Development Report 2016?...
    baseline done, sleeping...
    [Q1] saved. (1/10 done)
  [Q2] simple_fact: What year did WDR focus on governance and the law?...
    baseline done, sleeping...
    [Q2] saved. (2/10 done)
  [Q3] simple_fact: Which WDR report focused on the changing nature of work?...
    baseline done, sleeping...
    [Q3] saved. (3/10 done)
  [Q4] deep_context: How has the World Bank's approach to digital dividends evolv...
    baseline done, sleeping...
    [Q4] saved. (4/10 done)
  [Q5] deep_context: What specific policy recommendations does the WDR make for h...
    baseline done, sleeping...
    [Q5] saved. (5/10 done)
  [Q6] deep_context: How do WDR reports address the role of institutions in econo...
    baseline done, sleeping...
    [Q6] saved. (6/10 done)
  [Q7] ambiguous: What does the WDR say about poverty?...
    baseline done, sleeping...
    [Q7] saved. (7/10 done)
  [Q8] ambiguous: How does climate chang

In [13]:
# Cell 13: LLM judge scoring with partial progress saving
import json, re

JUDGE_CACHE   = os.path.join(CACHE_DIR, "scores.pkl")
JUDGE_PARTIAL = os.path.join(CACHE_DIR, "scores_partial.pkl")

def llm_judge(query: str, answer: str, chunks: list[dict]) -> dict:
    context_preview = format_context(chunks[:3])[:1500]
    prompt = f"""Evaluate this RAG answer on three criteria (score 1-5 each):

Query: {query}
Context (excerpt): {context_preview}
Answer: {answer}

Score and briefly justify each:
1. Correctness (factually accurate given context): X/5 — reason
2. Grounding (cites sources, stays in context): X/5 — reason
3. Completeness (addresses all parts of the query): X/5 — reason

Output as JSON: {{"correctness": int, "grounding": int, "completeness": int, "notes": str}}"""

    raw = call_gemini(prompt)
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {"correctness": 0, "grounding": 0, "completeness": 0, "notes": raw}
    except Exception:
        return {"correctness": 0, "grounding": 0, "completeness": 0, "notes": raw}


if os.path.exists(JUDGE_CACHE):
    with open(JUDGE_CACHE, "rb") as f:
        scored_results = pickle.load(f)
    print(f"Loaded {len(scored_results)} scored results from cache.")
else:
    scored_results = []
    if os.path.exists(JUDGE_PARTIAL):
        with open(JUDGE_PARTIAL, "rb") as f:
            scored_results = pickle.load(f)
        tqdm.write(f"Resuming judge scoring from partial: {len(scored_results)} done.")

    scored_ids = {r["id"] for r in scored_results}
    remaining  = [r for r in eval_results if r["id"] not in scored_ids]
    print(f"To score: {len(remaining)} remaining")

    for r in tqdm(remaining, desc="LLM judging"):
        tqdm.write(f"  Scoring {r['id']} baseline...")
        base_scores = llm_judge(r["query"], r["baseline"]["answer"], r["baseline"]["chunks"])
        time.sleep(FREE_TIER_DELAY)

        tqdm.write(f"  Scoring {r['id']} enhanced...")
        enha_scores = llm_judge(r["query"], r["enhanced"]["answer"], r["enhanced"]["chunks"])

        scored_results.append({**r, "baseline_scores": base_scores, "enhanced_scores": enha_scores})

        with open(JUDGE_PARTIAL, "wb") as f:
            pickle.dump(scored_results, f)
        tqdm.write(f"    [{r['id']}] scored and saved. ({len(scored_results)}/{len(eval_results)} done)")

    if len(scored_results) == len(eval_results):
        with open(JUDGE_CACHE, "wb") as f:
            pickle.dump(scored_results, f)
        tqdm.write("All scoring complete. Final cache saved.")

To score: 10 remaining


LLM judging:   0%|          | 0/10 [00:00<?, ?it/s]

  Scoring Q1 baseline...
  Scoring Q1 enhanced...
    [Q1] scored and saved. (1/10 done)
  Scoring Q2 baseline...
  Scoring Q2 enhanced...
    [Q2] scored and saved. (2/10 done)
  Scoring Q3 baseline...
  Scoring Q3 enhanced...
    [Q3] scored and saved. (3/10 done)
  Scoring Q4 baseline...
  Scoring Q4 enhanced...
    [Q4] scored and saved. (4/10 done)
  Scoring Q5 baseline...
  Scoring Q5 enhanced...
    [Q5] scored and saved. (5/10 done)
  Scoring Q6 baseline...
  Scoring Q6 enhanced...
    [Q6] scored and saved. (6/10 done)
  Scoring Q7 baseline...
  Scoring Q7 enhanced...
    [Q7] scored and saved. (7/10 done)
  Scoring Q8 baseline...
  Scoring Q8 enhanced...
    [Q8] scored and saved. (8/10 done)
  Scoring Q9 baseline...
  Scoring Q9 enhanced...
    [Q9] scored and saved. (9/10 done)
  Scoring Q10 baseline...
  Scoring Q10 enhanced...
    [Q10] scored and saved. (10/10 done)
All scoring complete. Final cache saved.


In [14]:
# Cell 14: Retrieval precision — check if retrieved chunks are relevant by year/topic
def retrieval_precision(chunks: list[dict], query: str):
    keywords = set(re.findall(r'\b\w{4,}\b', query.lower()))
    relevant = 0
    for c in chunks:
        chunk_words = set(c["text"].lower().split())
        overlap = len(keywords & chunk_words) / max(len(keywords), 1)
        if overlap >= 0.15:
            relevant += 1
    return relevant / len(chunks) if chunks else 0.0

retrieval_rows = []
for r in scored_results:
    retrieval_rows.append({
        "id":                  r["id"],
        "type":                r["type"],
        "baseline_precision":  retrieval_precision(r["baseline"]["chunks"], r["query"]),
        "enhanced_precision":  retrieval_precision(r["enhanced"]["chunks"],  r["query"]),
    })

retrieval_df = pd.DataFrame(retrieval_rows)
print(retrieval_df.to_string(index=False))
print(f"\nAvg baseline precision : {retrieval_df['baseline_precision'].mean():.3f}")
print(f"Avg enhanced precision : {retrieval_df['enhanced_precision'].mean():.3f}")

 id         type  baseline_precision  enhanced_precision
 Q1  simple_fact                 1.0                 1.0
 Q2  simple_fact                 0.6                 1.0
 Q3  simple_fact                 0.6                 1.0
 Q4 deep_context                 1.0                 1.0
 Q5 deep_context                 0.8                 1.0
 Q6 deep_context                 1.0                 1.0
 Q7    ambiguous                 0.8                 0.8
 Q8    ambiguous                 1.0                 1.0
 Q9    edge_case                 0.2                 0.2
Q10    edge_case                 0.0                 0.4

Avg baseline precision : 0.700
Avg enhanced precision : 0.840


In [15]:
# Cell 15: Build and save comprehensive results DataFrame
rows = []
for r in scored_results:
    for pipe, key in [("baseline", "baseline_scores"), ("enhanced", "enhanced_scores")]:
        s = r[key]
        rows.append({
            "Query ID":     r["id"],
            "Type":         r["type"],
            "Pipeline":     pipe,
            "Correctness":  s.get("correctness", 0),
            "Grounding":    s.get("grounding",   0),
            "Completeness": s.get("completeness",0),
            "Avg Score":    round(np.mean([s.get("correctness",0), s.get("grounding",0), s.get("completeness",0)]), 2),
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(OUTPUT_DIR, "evaluation_results.csv"), index=False)

summary = results_df.groupby("Pipeline")[["Correctness","Grounding","Completeness","Avg Score"]].mean().round(3)
print("="*55)
print("           EVALUATION SUMMARY (avg scores, 1–5 scale)")
print("="*55)
print(summary.to_string())
print("="*55)

           EVALUATION SUMMARY (avg scores, 1–5 scale)
          Correctness  Grounding  Completeness  Avg Score
Pipeline                                                 
baseline          4.5        3.8           4.9        4.4
enhanced          4.5        3.3           4.8        4.2


In [16]:
# Cell 16: Demo log
demo_log = []
for r in scored_results[:4]:
    demo_log.append({
        "query_id":         r["id"],
        "query_type":       r["type"],
        "query":            r["query"],
        "baseline_answer":  r["baseline"]["answer"],
        "baseline_sources": [(c["year"], c["page"]) for c in r["baseline"]["chunks"]],
        "baseline_scores":  r["baseline_scores"],
        "enhanced_answer":  r["enhanced"]["answer"],
        "enhanced_sources": [(c["year"], c["page"]) for c in r["enhanced"]["chunks"]],
        "enhanced_scores":  r["enhanced_scores"],
    })

with open(os.path.join(OUTPUT_DIR, "demo_log.json"), "w") as f:
    json.dump(demo_log, f, indent=2)

#print first demo entry
d = demo_log[0]
print(f"{'='*60}")
print(f"QUERY [{d['query_id']} | {d['query_type']}]: {d['query']}")
print(f"{'─'*60}")
print(f"BASELINE ANSWER:\n{d['baseline_answer'][:500]}...")
print(f"Sources: {d['baseline_sources']}")
print(f"Scores: {d['baseline_scores']}")
print(f"{'─'*60}")
print(f"ENHANCED ANSWER:\n{d['enhanced_answer'][:500]}...")
print(f"Sources: {d['enhanced_sources']}")
print(f"Scores: {d['enhanced_scores']}")
print(f"{'='*60}")

QUERY [Q1 | simple_fact]: What was the theme of the World Development Report 2016?
────────────────────────────────────────────────────────────
BASELINE ANSWER:
The provided context does not contain information regarding the theme of the World Development Report 2016....
Sources: [(2018, 15), (2020, 17), (2021, 15), (2025, 19), (2025, 1)]
Scores: {'correctness': 5, 'grounding': 5, 'completeness': 5, 'notes': 'The model correctly identified that the provided context (which discusses WDR 2018) does not contain the answer to the query about WDR 2016. It accurately reported the absence of information rather than hallucinating an answer.'}
────────────────────────────────────────────────────────────
ENHANCED ANSWER:
(1) **Direct Answer:** The theme of the World Development Report 2016 is "Digital Dividends." It explores the impact of the internet, mobile phones, and related technologies on economic development, specifically examining how these technologies can promote growth, jobs, and serv